# Polar Twist: mPCN vs P Independent Chains

Compare one mPCN chain with P proposals to P independent mPCN chains with one proposal. The single-proposal chains are thinned every P steps to match likelihood-evaluation budgets.

In [2]:
import json
import hashlib
from pathlib import Path
import time

import numpy as np
import matplotlib.pyplot as plt

from multiproposal.algorithms.mpcn import mpcn_chain
from multiproposal.algorithms.effective_sample_size import estimate_effective_sample_size
from multiproposal.problems.toy_custom_likelihood import ToyCustomLikelihood2D

In [3]:
def f_polar_twist(x, alpha):
    x1, x2 = x
    radius = np.sqrt(x1 ** 2 + x2 ** 2)
    comp1 = x1 * np.cos(alpha * radius) - x2 * np.sin(alpha * radius)
    comp2 = x1 * np.sin(alpha * radius) + x2 * np.cos(alpha * radius)
    return np.array([comp1, comp2])


def log_likelihood_polar_twist(x, y_obs, sigma=0.3, alpha=0.5):
    r = f_polar_twist(x, alpha=alpha) - y_obs
    return -0.5 * np.dot(r, r) / (sigma ** 2)

In [4]:
# Data configuration
alpha = 2.0
sigma_noise = 1.0
prior_std = 2.0
prior_cov = prior_std ** 2 * np.array([[1.0, 0.3], [0.3, 0.5]])
prior_mean = np.zeros(2)
seed_data = 202

rng_data = np.random.default_rng(seed_data)
prior_sample = rng_data.multivariate_normal(prior_mean, prior_cov)
theta_true = f_polar_twist(prior_sample, alpha=alpha)
y_obs = theta_true + rng_data.normal(0.0, sigma_noise, size=theta_true.shape)

def log_likelihood(x):
    return log_likelihood_polar_twist(x, y_obs, sigma=sigma_noise, alpha=alpha)

problem = ToyCustomLikelihood2D(
    log_likelihood_fn=log_likelihood,
    prior_mean=prior_mean,
    prior_cov=prior_cov,
)

print('y_obs:', y_obs)
print('True x:', prior_sample)

y_obs: [ 1.59110063 -3.32997972]
True x: [-3.13349585 -2.43303473]


In [5]:
# Experiment configuration
P = 10
n_iters = 50000
rho = 0.1
burn_in = 500
seed_mpcn = 202
seed_single_base = 1000

# Parallel options for the P-proposal chain
n_jobs = None  # None or <=0 uses core_frac cap; 1 disables parallelism
core_frac = 0.7
parallel_backend = 'auto'  # 'auto' -> threads, 'process' for multiprocessing

n_iters_single = n_iters * P
thin_stride = P

print('P:', P)
print('n_iters (P-proposal):', n_iters)
print('n_iters_single (1-proposal):', n_iters_single)
print('thin_stride:', thin_stride)

P: 10
n_iters (P-proposal): 50000
n_iters_single (1-proposal): 500000
thin_stride: 10


In [6]:
# Run mPCN with P proposals
rng_mpcn = np.random.default_rng(seed_mpcn)
x0 = problem.sample_prior(rng_mpcn)

t0 = time.perf_counter()
chain_mpcn = mpcn_chain(
    x0,
    problem,
    rng_mpcn,
    n_iters,
    rho=rho,
    n_props=P,
    n_jobs=n_jobs,
    core_frac=core_frac,
    parallel_backend=parallel_backend,
)
runtime_mpcn_sec = time.perf_counter() - t0

print(f'mPCN P-proposal runtime (sec): {runtime_mpcn_sec:.2f}')

mPCN P-proposal runtime (sec): 9.58


In [7]:
# Run P independent mPCN chains with 1 proposal, then thin
chains_single = np.zeros((P, n_iters_single + 1, problem.dim), dtype=float)
for p in range(P):
    seed_chain = seed_single_base + p
    rng_chain = np.random.default_rng(seed_chain)
    x0_chain = problem.sample_prior(rng_chain)
    chain = mpcn_chain(
        x0_chain,
        problem,
        rng_chain,
        n_iters_single,
        rho=rho,
        n_props=1,
    )
    chains_single[p] = chain
    print(f'Finished chain {p + 1}/{P}')

thin_indices = np.arange(0, n_iters_single + 1, thin_stride)
chains_single_thin = chains_single[:, thin_indices]
print('Thinned single-proposal chains shape:', chains_single_thin.shape)

Finished chain 1/10
Finished chain 2/10
Finished chain 3/10
Finished chain 4/10
Finished chain 5/10
Finished chain 6/10
Finished chain 7/10
Finished chain 8/10
Finished chain 9/10
Finished chain 10/10
Thinned single-proposal chains shape: (10, 50001, 2)


In [8]:
def compute_msjd_per_param(chain):
    if chain.shape[0] < 2:
        return np.zeros(chain.shape[1])
    jumps = np.diff(chain, axis=0)
    return np.mean(jumps * jumps, axis=0)


def compute_ess_per_param(chain, max_lag=500):
    if chain.shape[0] < 2:
        return np.zeros(chain.shape[1])
    variances = np.var(chain, axis=0)
    if np.all(variances == 0):
        return np.zeros(chain.shape[1])
    ess_vals = estimate_effective_sample_size(chain, max_lag=max_lag)
    ess_vals = np.asarray(ess_vals, dtype=float)
    ess_vals[variances == 0] = 0.0
    return ess_vals


def compute_rhat(chains):
    # chains shape: (n_chains, n_samples, dim)
    n_chains, n_samples, dim = chains.shape
    if n_samples < 2:
        return np.full(dim, np.nan)
    chain_means = np.mean(chains, axis=1)
    chain_vars = np.var(chains, axis=1, ddof=1)
    mean_of_means = np.mean(chain_means, axis=0)
    B = n_samples * np.mean((chain_means - mean_of_means) ** 2, axis=0)
    W = np.mean(chain_vars, axis=0)
    var_hat = (n_samples - 1) / n_samples * W + B / n_samples
    return np.sqrt(var_hat / W)


def summarize_metrics(chain, label, max_lag=500):
    post = chain[burn_in:]
    ess_vals = compute_ess_per_param(post, max_lag=max_lag)
    msjd_vals = compute_msjd_per_param(post)
    return {
        'label': label,
        'ess_mean': float(np.nanmean(ess_vals)),
        'msjd_mean': float(np.nanmean(msjd_vals)),
    }

metrics_mpcn = summarize_metrics(chain_mpcn, f'mPCN P={P}')
metrics_single = [summarize_metrics(chains_single_thin[p], f'chain_{p}') for p in range(P)]
rhat_single = compute_rhat(chains_single_thin[:, burn_in:])

print('mPCN P-proposal metrics:', metrics_mpcn)
print('Single-proposal chains mean ESS:', np.mean([m['ess_mean'] for m in metrics_single]))
print('Single-proposal chains mean MSJD:', np.mean([m['msjd_mean'] for m in metrics_single]))
print('Rhat (single-proposal chains):', rhat_single)

post_mpcn = chain_mpcn[burn_in:]
post_single = chains_single_thin[:, burn_in:].reshape(-1, problem.dim)
mean_mpcn = np.mean(post_mpcn, axis=0)
mean_single = np.mean(post_single, axis=0)

print('Posterior mean (mPCN):', mean_mpcn)
print('Posterior mean (P chains):', mean_single)
print('Mean error to true x (mPCN):', np.linalg.norm(mean_mpcn - prior_sample))
print('Mean error to true x (P chains):', np.linalg.norm(mean_single - prior_sample))

Estimating ESS for each parameter.
Estimating ESS for each parameter.
Estimating ESS for each parameter.
Estimating ESS for each parameter.
Estimating ESS for each parameter.
Estimating ESS for each parameter.
Estimating ESS for each parameter.
Estimating ESS for each parameter.
Estimating ESS for each parameter.
Estimating ESS for each parameter.
Estimating ESS for each parameter.
mPCN P-proposal metrics: {'label': 'mPCN P=10', 'ess_mean': 11156.246007279216, 'msjd_mean': 3.590802663308903}
Single-proposal chains mean ESS: 12125.28019011116
Single-proposal chains mean MSJD: 3.9033463467920897
Rhat (single-proposal chains): [1.00002304 1.00001224]
Posterior mean (mPCN): [ 0.02196708 -0.33910277]
Posterior mean (P chains): [-0.02874677 -0.35668162]
Mean error to true x (mPCN): 3.787016942755841
Mean error to true x (P chains): 3.7350647957266823


In [9]:
# Save results to estimations folder
estimations_root = Path('estimations')
estimations_root.mkdir(parents=True, exist_ok=True)

config = {
    'model': 'polar_twist',
    'algorithm': 'mpcn',
    'P': P,
    'n_iters': n_iters,
    'n_iters_single': n_iters_single,
    'thin_stride': thin_stride,
    'rho': rho,
    'seed_data': seed_data,
    'seed_mpcn': seed_mpcn,
    'seed_single_base': seed_single_base,
    'alpha': alpha,
    'sigma_noise': sigma_noise,
    'prior_std': prior_std,
    'prior_cov': prior_cov.tolist(),
    'burn_in': burn_in,
    'n_jobs': n_jobs,
    'core_frac': core_frac,
    'parallel_backend': parallel_backend
}

config_json = json.dumps(config, sort_keys=True)
config_hash = hashlib.sha256(config_json.encode('utf-8')).hexdigest()[:12]
run_name = f'polar_twist_mpcn_P{P}_rho{rho}_iters{n_iters}_seed{seed_mpcn}_h{config_hash}'
run_dir = estimations_root / 'polar_twist' / 'mpcn' / run_name
run_dir.mkdir(parents=True, exist_ok=True)

with open(run_dir / 'config.json', 'w', encoding='utf-8') as f:
    f.write(config_json)

np.savez_compressed(
    run_dir / 'samples.npz',
    chain_mpcn=chain_mpcn,
    chains_single=chains_single,
    chains_single_thin=chains_single_thin,
    prior_sample=prior_sample,
    y_obs=y_obs
)

metrics = {
    'mpcn': metrics_mpcn,
    'single_chains': metrics_single,
    'rhat_single': rhat_single.tolist(),
    'runtime_mpcn_sec': runtime_mpcn_sec
}

with open(run_dir / 'metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

print('Saved run to:', run_dir)

Saved run to: estimations/polar_twist/mpcn/polar_twist_mpcn_P10_rho0.1_iters50000_seed202_h5370bc33a548
